# Latihan 2: Agen Multi-Tool dengan MCP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dteti-workshop/workshop-genai-migas/blob/main/colab/hari4/09-mcp-latihan-2.ipynb)

Ini versi **Google Colab**. Isi dan urutan contohnya sama dengan versi VS Code; yang berbeda hanya cell Persiapan di bawah, yang menyalin repository, memasang pustaka, dan mengisi API key di runtime Colab.

1. Pilih **File > Save a copy in Drive** agar perubahan tersimpan di akun Anda sendiri.
2. Jalankan tiga cell **Persiapan Colab** berurutan (sekitar 2-3 menit pada sesi baru).
3. Kerjakan dari atas ke bawah dengan `Shift+Enter`.

Kode jawaban latihan ada di bagian **Kode Jawaban Latihan** pada [PDF bab ini](https://github.com/dteti-workshop/workshop-genai-migas/blob/main/hari4/09-mcp/09-mcp.pdf). Salin kodenya ke cell yang bertanda `TODO`.

Kalau runtime terputus, file di `/content` (termasuk `.env`) hilang. Jalankan ulang Persiapan, lalu lanjutkan dari cell yang sedang Anda kerjakan.

## Persiapan Colab
Jalankan tiga cell berikut berurutan sebelum cell lain di notebook ini.

In [ ]:
# Persiapan 1 - Menyalin repository dan memasang pustaka (sekitar 2-3 menit pada sesi baru)
import os
import subprocess
import sys

REPO_URL = "https://github.com/dteti-workshop/workshop-genai-migas.git"
REPO = "/content/workshop-genai-migas" if os.path.isdir("/content") else os.path.abspath("workshop-genai-migas")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)

pasang = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "colab/requirements-colab.txt", "-e", "."],
                        capture_output=True, text=True)
if pasang.returncode != 0:
    print(pasang.stderr[-3000:])
    raise RuntimeError("Instalasi pustaka gagal. Jalankan ulang cell ini.")

# "-e ." memasang common dan migas_shared agar subproses (server MCP) dapat mengimpornya; kernel yang sedang berjalan
# belum memuat instalasi baru, jadi folder repo juga ditambahkan ke sys.path
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repository dan pustaka siap. Folder kerja:", os.getcwd())

In [ ]:
# Persiapan 2 - Mengisi ID peserta dan API key (dilewati bila .env sudah berisi key)
import re
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv

env = Path(REPO) / ".env"


def rahasia(nama: str):
    """Nilai dari Colab Secrets (ikon kunci di panel kiri), atau None bila tidak ada."""
    try:
        from google.colab import userdata
        return userdata.get(nama)
    except Exception:  # noqa: BLE001 - bukan di Colab, secret belum dibuat, atau akses notebook belum diizinkan
        return None


if not (env.exists() and re.search(r"(?m)^ONLINE_API_KEY=(?!isi-dengan)\S+", env.read_text(encoding="utf-8"))):
    peserta = (rahasia("PARTICIPANT_ID") or input("ID peserta (nomor meja dan nama singkat, contoh P07-andi): ")).strip()
    kunci = (rahasia("ONLINE_API_KEY") or getpass("API key workshop (ketikan tidak ditampilkan): ")).strip()
    if not kunci:
        raise ValueError("API key belum diisi. Jalankan ulang cell ini.")
    isi = (Path(REPO) / ".env.example").read_text(encoding="utf-8")
    for nama, nilai in (("PARTICIPANT_ID", peserta or "P00-nama"), ("ONLINE_API_KEY", kunci)):
        isi = re.sub(rf"(?m)^{nama}=.*$", lambda _, n=nama, v=nilai: f"{n}={v}", isi)
    env.write_text(isi, encoding="utf-8")
    del kunci

# Nilai dari .env dimuat ulang; modul yang sudah diimpor dibuang agar membaca nilai terbaru
load_dotenv(env, override=True)
for modul in [m for m in sys.modules if m.split(".")[0] in ("common", "migas_shared", "rag_pipeline")]:
    del sys.modules[modul]
print("File .env siap untuk peserta", os.environ.get("PARTICIPANT_ID"))

In [ ]:
# Persiapan 3 - Memeriksa koneksi ke model
from migas_shared.llm import chat

try:
    print("Jawaban model:", chat("Balas hanya dengan kata: SIAP").strip())
except Exception as error:  # noqa: BLE001
    raise RuntimeError("Model tidak menjawab. Periksa API key (jalankan ulang Persiapan 2) dan koneksi internet. "
                       f"Pesan asli: {error}") from error

Isi bagian bertanda `TODO(...)` dengan kode dari bagian **Kode Jawaban Latihan** pada PDF bab ini, lalu jalankan sel satu per satu
dengan Shift+Enter. Baris `... = None  # TODO` atau `raise NotImplementedError` berarti kode itu harus Anda ganti.

Menggabungkan tool dari server MCP (lapangan, stdio) dengan tool yang Anda tulis sendiri di Python,
dalam SATU agen. Agen memilih tool dari deskripsinya, tidak peduli dari mana tool itu berasal.

- Tool MCP: hitung_water_cut, konversi_satuan, cek_status_sumur (`field_stdio_server.py`)
- Tool lokal: hitung_kerugian_produksi (ditulis di file ini)

Kode async dijalankan lewat `jalankan(...)` dari `migas_shared.tampil`, yang bekerja di skrip biasa maupun
di Jupyter/Interactive Window.

In [ ]:
# Langkah 1 - Impor dan model
import os
import sys

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from migas_shared.config import ROOT
from migas_shared.tampil import jalankan

load_dotenv()

PATH_SERVER = ROOT / "hari4" / "09-mcp" / "code" / "servers" / "field_stdio_server.py"

model = ChatOpenAI(
    model=os.getenv("ONLINE_MODEL"),
    base_url=os.getenv("ONLINE_BASE_URL"),
    api_key=os.getenv("ONLINE_API_KEY"),
    reasoning_effort=os.getenv("LLM_REASONING_EFFORT", "none"),
)

## Langkah 2 - Tool lokal
Skema masukan Pydantic dan tool `hitung_kerugian_produksi`, ditulis langsung di Python.

In [ ]:
class KerugianInput(BaseModel):
    """Skema masukan tool kerugian produksi."""

    # TODO(B9.4): definisikan field KerugianInput: downtime_jam dan laju_normal_bopd (float, keduanya >= 0, dengan description)
    pass  # TODO: definisikan field


# TODO(B9.5): buat tool hitung_kerugian_produksi yang mencetak panggilannya, lalu menghitung barel hilang = laju_normal_bopd * downtime_jam / 24
hitung_kerugian_produksi = None  # TODO: ganti dengan implementasi Anda

## Langkah 3 - Menggabungkan tool MCP dan tool lokal
Klien membuka sesi per panggilan dan menutupnya sendiri, jadi tidak ada yang perlu ditutup antar sel.

In [ ]:
# TODO(B9.6): buat MultiServerMCPClient untuk satu server stdio "lapangan" yang menjalankan PATH_SERVER dengan sys.executable
client = None  # TODO: ganti dengan implementasi Anda
# TODO(B9.7): ambil tool MCP dengan jalankan(client.get_tools()), lalu gabungkan dengan tool lokal menjadi semua_tool
tools_mcp = []  # TODO: ganti dengan implementasi Anda
semua_tool = []  # TODO: ganti dengan implementasi Anda
print(f"Tool MCP:   {[t.name for t in tools_mcp]}")
print(f"Tool lokal: {[hitung_kerugian_produksi.name]}\n")

## Langkah 4 - Membuat agen
Satu agen menerima semua tool, tanpa peduli asalnya.

In [ ]:
# TODO(B9.8): buat agen dari model dan semua_tool dengan system prompt asisten teknis lapangan (pakai tool, jangan menghitung sendiri)
agent = None  # TODO: ganti dengan implementasi Anda


def tanya_agen(pertanyaan: str) -> None:
    print(f"Pengguna: {pertanyaan}")
    response = jalankan(agent.ainvoke({"messages": [("human", pertanyaan)]}))
    panggilan = [c["name"] for m in response["messages"] for c in getattr(m, "tool_calls", [])]
    print(f"   Tool yang dipanggil: {panggilan}")
    print(f"Agen: {response['messages'][-1].content}\n")

## Langkah 5 - Pertanyaan 1: memakai tool MCP

In [ ]:
tanya_agen("Berapa water cut sumur dengan produksi 95 bopd minyak dan 380 bwpd air?")  # tool MCP

## Langkah 6 - Pertanyaan 2: memakai tool lokal

In [ ]:
tanya_agen("Sumur SR-020 mati 18 jam dengan laju normal 120 bopd. Berapa barel minyak yang hilang?")  # tool lokal

## Langkah 7 - Pertanyaan 3: memakai keduanya

In [ ]:
tanya_agen("Cek status sumur SR-027, lalu hitung kehilangan produksinya bila ia mati 10 jam pada laju normalnya.")  # keduanya